# COMP663 Assignment 2 — Classical Optimisation

**Student ID:** 1173808  
**Dataset:** `forest_cover_data.csv`  
**Primary metric:** macro-F1


## ENV & Libs Setup


In [ ]:
from pathlib import Path
import ast
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import (
    ParameterSampler,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
DEVICE = torch.device("cpu")
torch.set_num_threads(4)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "forest_cover_data.csv"
FIGURE_PATH = ROOT / "figures" / "performance_comparison.png"
MODEL_PATH = ROOT / "models" / "1173808_Assignment2_final.pt"

TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
SEARCH_EPOCHS = 20
FINAL_EPOCHS = 100
RANDOM_TRIALS = 6
BAYESIAN_TRIALS = 8
NAS_TRIALS = 8

print(f"env ready: torch={torch.__version__}, device={DEVICE}, seed={SEED}")

## Task 1 — Baseline model and candidate hyperparameters

### 1.1 Preprocessing pipeline and feature engineering

As we found during assginment 1:

| Decision                 | Evidence from EDA                                                                                                                      | Action                                                                                                       | Reason / trade-off                                                                                                                                                                                              |
| ------------------------ | -------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Missing values           | Zero missing values across all 15 columns (confirmed in Task 1.1)                                                                      | pass through                                                                                                 | Imputing when there is nothing to impute would add unnecessary complexity and risk introducing artificial patterns.                                                                                             |
| Scaling / transformation | Distance features have large ranges and outliers (Task 1.3)                                                                            | Apply `StandardScaler` to the 10 continuous features and leave the 4 binary Wilderness_Area columns unscaled | Scaling puts continuous features on a comparable range. The fitted transformation stays inside the validation pipeline, which prevents leakage. Some models are notdistance-sensitive which will discuss later. |
| Feature engineering      | Elevation, distance features, and wilderness area already show useful class separation; wilderness columns are already one-hot encoded | No additional feature engineering                                                                            | The existing features already contain useful information. Adding polynomial features would increase complexity without EDA evidence that they are needed.                                                       |
| Class imbalance          | The largest and smallest classes have a 31:1 ratio (Task 1.2)                                                                          | Use class weighting and evaluate with macro-F1                                                               | Class weighting gives more importance to rare classes without changing the training data. Macro-F1 makes minority-class performance visible.                                                                    |




#### Load data and check data integrity


In [ ]:
# load data from CSV file
data = pd.read_csv(DATA_PATH)
target = "Cover_Type"

# filter out rows with missing values in the target column
data = data.dropna(subset=[target])
feature_names = [column for column in data.columns if column != target]
continuous_features = [
    column for column in feature_names if not column.startswith("Wilderness_Area")
]

# check data integrity
assert data.shape == (571_012, 15), data.shape
assert len(feature_names) == 14
assert set(data[target].unique()) == {1, 2, 3, 4, 5}
assert data.isna().sum().sum() == 0

# print data shape, feature names, and target value counts with percentages
print("Shape:", data.shape)
print("Features:", feature_names)
display(
    data[target]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
    .assign(percentage=lambda frame: 100 * frame["count"] / len(data))
)

#### Data set split

In [ ]:
# split data into training, validation, and test sets
train_validation_frame, test_frame = train_test_split(
    data, test_size=TEST_FRACTION, stratify=data[target], random_state=SEED
)

train_frame, validation_frame = train_test_split(
    train_validation_frame,
    test_size=VALIDATION_FRACTION / (TRAIN_FRACTION + VALIDATION_FRACTION),
    stratify=train_validation_frame[target],
    random_state=SEED,
)

# validate the size of the splits and print the number of samples in each set
assert len(train_frame) + len(validation_frame) + len(test_frame) == len(data)
print(
    f"Train / validation / test: {len(train_frame):,} / {len(validation_frame):,} / {len(test_frame):,}"
)

### 1.2 Primary and secondary evaluation metrics

As this dataset have a strong imblanaced class,

**Macro-F1** will be the primary metric to cover type has equal importance.

**Balanced accuracy** is the secondary metric because it reflect the model recalls on each class equally.


### 1.3 Baseline model training and evaluation procedure

- (1) Initial the baseline mode with configuration in assignment requirement.

- (2) traning baseline model and compute metrics on validation dataset.


In [ ]:
# Define the fixed baseline architecture required by the assignment.
class BaselineNN(nn.Module):
    """The architecture supplied in baselineNN.ipynb."""

    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 24),  # 14 input features to 24 hidden units
            nn.Sigmoid(),               # required hidden-layer activation
            nn.Linear(24, 12),          # second hidden layer
            nn.Sigmoid(),               # required hidden-layer activation
            nn.Linear(12, 5),           # one logit for each Cover_Type class
        )

    def forward(self, x):
        return self.layers(x)


# Create a fresh baseline model on the selected device.
baseline_model = BaselineNN(len(feature_names)).to(DEVICE)
print(baseline_model)
print(
    f"Parameters: {sum(parameter.numel() for parameter in baseline_model.parameters()):,}"
)
# Check the supplied architecture: weights and biases total 725 trainable parameters.
assert (
    sum(parameter.numel() for parameter in baseline_model.parameters()) == 725
), "Parameter count mismatch"


In [ ]:
# Train the baseline model and evaluate it on the validation split.
BASELINE_CONFIG = {
    "learning_rate": 1e-3,
    "batch_size": 512,
    "weight_decay": 1e-4,
    "epochs": SEARCH_EPOCHS,
}

# Fit scaling values on training data only to prevent data leakage.
scaler = StandardScaler().fit(train_frame[continuous_features])


def prepare_baseline_data(frame):
    # Keep the original 14 feature columns and use float32 for PyTorch.
    features = frame[feature_names].astype("float32").copy()
    # Scale only continuous features; Wilderness_Area columns are already 0/1.
    features[continuous_features] = scaler.transform(features[continuous_features])
    # Change class labels from 1–5 to the 0–4 indices required by CrossEntropyLoss.
    return features.to_numpy(), frame[target].to_numpy(dtype=np.int64) - 1


# Apply the training-fitted scaler to both training and validation features.
x_train, y_train = prepare_baseline_data(train_frame)
x_validation, y_validation = prepare_baseline_data(validation_frame)

# Give rare classes larger loss weights so all five classes affect training.
class_counts = np.bincount(y_train, minlength=5)
class_weights = len(y_train) / (5 * class_counts)
loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)
# Adam updates the baseline parameters using the fixed Task 1 settings.
optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=BASELINE_CONFIG["learning_rate"],
    weight_decay=BASELINE_CONFIG["weight_decay"],
)

# Move the training arrays to PyTorch tensors once before the epoch loop.
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
# Use the fixed seed so the mini-batch order is reproducible.
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

# Training mode enables gradient calculation and parameter updates.
baseline_model.train()
for _ in range(BASELINE_CONFIG["epochs"]):
    # Shuffle the training rows once per epoch.
    order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
    for start in range(0, len(order), BASELINE_CONFIG["batch_size"]):
        batch_index = order[start : start + BASELINE_CONFIG["batch_size"]]
        optimizer.zero_grad()  # clear gradients from the previous mini-batch
        loss = loss_fn(
            baseline_model(x_train_tensor[batch_index]), y_train_tensor[batch_index]
        )
        loss.backward()  # compute gradients by backpropagation
        optimizer.step()  # update weights and biases

# Evaluation mode and no_grad disable training updates for validation.
baseline_model.eval()
with torch.no_grad():
    validation_logits = baseline_model(
        torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)
    )
# The largest logit is the predicted class; return predictions to NumPy for metrics.
validation_prediction = validation_logits.argmax(dim=1).cpu().numpy()

validation_macro_f1 = f1_score(y_validation, validation_prediction, average="macro")
validation_balanced_accuracy = balanced_accuracy_score(
    y_validation, validation_prediction
)
# Metrics must be valid scores between zero and one.
assert 0 <= validation_macro_f1 <= 1
print("Validation macro-F1:", validation_macro_f1)
print("Validation balanced accuracy:", validation_balanced_accuracy)

### 1.4 Unoptimised baseline performance using held-out test data


In [ ]:
# evaluate the unoptimised baseline on the held-out test set
x_test, y_test = prepare_baseline_data(test_frame)

baseline_model.eval()
with torch.no_grad():
    test_logits = baseline_model(
        torch.tensor(x_test, dtype=torch.float32, device=DEVICE)
    )
test_prediction = test_logits.argmax(dim=1).cpu().numpy()

test_macro_f1 = f1_score(y_test, test_prediction, average="macro")
test_balanced_accuracy = balanced_accuracy_score(y_test, test_prediction)
assert 0 <= test_macro_f1 <= 1
print("Test macro-F1:", test_macro_f1)
print("Test balanced accuracy:", test_balanced_accuracy)


### 1.5 Candidate hyperparameters

The candidate hyperparameters are learning rate, batch size, weight decay, epochs, and, for NAS only, hidden-layer count, width, and activation. They may change convergence, regularisation, training cost, or model capacity.


### 1.6 Hyperparameter types and search ranges

| Hyperparameter | Type and range | Expected effect / cost |
|---|---|---|
| Learning rate | log continuous, 0.0003–0.003 | Controls update size; too large can be unstable. |
| Batch size | categorical: 256, 512, 1024 | Larger batches use fewer updates per epoch. |
| Weight decay | log continuous, 0.000001–0.001 | Regularises weights; too much can underfit. |
| Epochs | fixed at 20 per trial | Keeps the comparison within a practical budget. |
| Hidden layers, widths, activation | NAS only | Changes capacity and parameter count. |


## Task 2 — Classical hyperparameter search


### 2.1 Selected classical search method and justification

I use random search. It covers the learning-rate and weight-decay ranges more efficiently than a small grid because both ranges are meaningful on a log scale.


### 2.2 Hyperparameters to optimise

Random search optimises learning rate, batch size, and weight decay. These are the Task 1 candidates; the supplied baseline architecture stays fixed.


### 2.3 Computational budget and justification

The budget is six trials, each trained for 20 epochs. This is small enough for CPU execution while testing different training settings.


### 2.4 Apply the selected search method


In [ ]:
# Define reusable model and training functions for every optimisation trial.
BASELINE_ARCHITECTURE = {"hidden_layers": [24, 12], "activation": "Sigmoid"}


class SearchMLP(nn.Module):
    """Build a NAS candidate from its declared hidden layers and activation."""

    def __init__(self, input_size, hidden_layers, activation):
        super().__init__()
        activation_layer = getattr(nn, activation)
        layers = []
        previous_width = input_size
        for width in hidden_layers:
            layers.extend([nn.Linear(previous_width, width), activation_layer()])
            previous_width = width
        layers.append(nn.Linear(previous_width, 5))  # five Cover_Type logits
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)


def build_model(architecture):
    # Keep the supplied baseline class for its exact 24-12-Sigmoid architecture.
    if architecture == BASELINE_ARCHITECTURE:
        return BaselineNN(len(feature_names)).to(DEVICE)
    return SearchMLP(len(feature_names), **architecture).to(DEVICE)


def fit_and_evaluate(config, architecture, fit_frame, evaluate_frame):
    # Reset PyTorch so every trial starts from a reproducible initial state.
    torch.manual_seed(SEED)
    model = build_model(architecture)

    # Fit one scaler on this trial's training rows only, then transform both splits.
    trial_scaler = StandardScaler().fit(fit_frame[continuous_features])

    def transform(frame):
        values = frame[feature_names].astype("float32").copy()
        values[continuous_features] = trial_scaler.transform(values[continuous_features])
        labels = frame[target].to_numpy(dtype=np.int64) - 1
        return values.to_numpy(), labels

    x_fit, y_fit = transform(fit_frame)
    x_evaluate, y_evaluate = transform(evaluate_frame)

    # Weight classes from the training split so rare classes contribute to the loss.
    class_counts = np.bincount(y_fit, minlength=5)
    class_weights = len(y_fit) / (5 * class_counts)
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(config["learning_rate"]),
        weight_decay=float(config["weight_decay"]),
    )

    # Convert training arrays once and use deterministic shuffled mini-batches.
    x_fit_tensor = torch.tensor(x_fit, dtype=torch.float32, device=DEVICE)
    y_fit_tensor = torch.tensor(y_fit, dtype=torch.long, device=DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    model.train()
    for _ in range(int(config["epochs"])):
        order = torch.randperm(len(x_fit_tensor), generator=generator, device=DEVICE)
        for start in range(0, len(order), int(config["batch_size"])):
            batch_index = order[start : start + int(config["batch_size"])]
            optimizer.zero_grad()
            loss = loss_fn(model(x_fit_tensor[batch_index]), y_fit_tensor[batch_index])
            loss.backward()
            optimizer.step()

    # Predict in chunks to keep evaluation memory bounded on CPU.
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(x_evaluate), 4096):
            batch = torch.tensor(x_evaluate[start : start + 4096], dtype=torch.float32, device=DEVICE)
            predictions.append(model(batch).argmax(dim=1).cpu().numpy())
    prediction = np.concatenate(predictions)
    metrics = {
        "macro_f1": f1_score(y_evaluate, prediction, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y_evaluate, prediction),
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "prediction": prediction,
        "target": y_evaluate,
    }
    return model, trial_scaler, metrics


# Random search samples configurations without evaluating every grid combination.
random_space = {
    "learning_rate": [3e-4, 1e-3, 3e-3],
    "batch_size": [256, 512, 1024],
    "weight_decay": [0.0, 1e-5, 1e-4],
}
random_rows = []
for trial_number, sampled in enumerate(
    ParameterSampler(random_space, n_iter=RANDOM_TRIALS, random_state=SEED), 1
):
    config = {**sampled, "epochs": SEARCH_EPOCHS}
    started = time.perf_counter()
    _, _, metrics = fit_and_evaluate(
        config, BASELINE_ARCHITECTURE, train_frame, validation_frame
    )
    random_rows.append({
        "trial": trial_number,
        **config,
        "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "parameters": metrics["parameters"],
        "seconds": time.perf_counter() - started,
    })

random_table = pd.DataFrame(random_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_random = random_table.iloc[0].to_dict()


### 2.5 Evaluation procedure and primary metric

Every trial trains on the same 60% training split, fits its scaler only on that split, and maximises validation macro-F1 on the same 20% validation split.


### 2.6 Best configuration, score, and search time


### 2.7 Search results table


## Task 3 — Bayesian optimisation


### 3.1 Hyperparameters to optimise

Bayesian optimisation uses the same Task 1 candidates as random search: learning rate, batch size, and weight decay.


### 3.2 Objective, search space, surrogate model, and acquisition process

The objective is validation macro-F1. Optuna's TPE sampler uses completed trials to model promising parameter values and proposes the next trial from promising regions.


### 3.3 Evaluation procedure and primary metric

Bayesian trials use the same training split, validation split, scaler rule, epoch budget, and macro-F1 objective as Tasks 1 and 2.


### 3.4 Computational budget and justification

The budget is eight trials of 20 epochs. Sequential feedback is useful even with a small budget.


### 3.5 Apply Bayesian optimisation


In [ ]:
# TPE is Optuna's Bayesian sampler. It uses earlier trials to choose later values.
bayesian_rows = []


def bayesian_objective(trial):
    # Sample only from the Task 1 ranges and keep the baseline architecture fixed.
    config = {
        "learning_rate": trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "epochs": SEARCH_EPOCHS,
    }
    started = time.perf_counter()
    _, _, metrics = fit_and_evaluate(
        config, BASELINE_ARCHITECTURE, train_frame, validation_frame
    )
    bayesian_rows.append({
        "trial": trial.number + 1,
        **config,
        "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "parameters": metrics["parameters"],
        "seconds": time.perf_counter() - started,
    })
    return metrics["macro_f1"]  # Optuna maximises validation macro-F1.


bayesian_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
bayesian_study.optimize(bayesian_objective, n_trials=BAYESIAN_TRIALS)
bayesian_table = pd.DataFrame(bayesian_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_bayesian = {**bayesian_study.best_params, "epochs": SEARCH_EPOCHS}


### 3.6 Best configuration, performance, and search time

The next cell reports the best Bayesian configuration, validation macro-F1, and total search time after execution.


In [ ]:
# Print the best Bayesian result after all eight sequential trials finish.
print("Best Bayesian configuration:", best_bayesian)
print("Best validation macro-F1:", bayesian_table.iloc[0]["macro_f1"])
print("Total Bayesian-search time:", bayesian_table["seconds"].sum())


### 3.7 Search results table

The table below lists every completed Bayesian trial.


In [ ]:
# Show every Bayesian trial, including the time used by each objective evaluation.
display(bayesian_table.round(6))


## Task 4 — Neural architecture search


### 4.1 NAS hyperparameters to optimise

NAS optimises hidden-layer count, hidden-layer widths, activation, learning rate, batch size, and weight decay. All choices are declared in Task 1.


### 4.2 Objective, NAS search space, and search strategy

I use fixed-budget random NAS. Each trial samples one architecture and training configuration, then maximises validation macro-F1.


### 4.3 Architecture design choices

The architecture search space allows one to three hidden layers, widths from 16 to 64, and Sigmoid or ReLU activation. The output remains five logits.


### 4.4 Evaluation procedure and primary metric

NAS uses the same 60% training split, 20% validation split, training-only scaling, 20-epoch budget, and macro-F1 metric as the other searches.


### 4.5 Computational budget and justification

The budget is eight NAS trials. It is enough to test real architecture choices without using an impractical exhaustive search.


### 4.6 Apply NAS to hyperparameters and architecture


In [ ]:
# Random NAS samples both an architecture and training settings from declared ranges.
rng = np.random.default_rng(SEED)
nas_rows = []
for trial_number in range(1, NAS_TRIALS + 1):
    # Sample genuine architecture choices: layer count, width, and activation.
    layer_count = int(rng.choice([1, 2, 3]))
    architecture = {
        "hidden_layers": [int(rng.choice([16, 24, 32, 48, 64])) for _ in range(layer_count)],
        "activation": str(rng.choice(["Sigmoid", "ReLU"])),
    }
    config = {
        "learning_rate": float(rng.choice([3e-4, 1e-3, 3e-3])),
        "batch_size": int(rng.choice([256, 512, 1024])),
        "weight_decay": float(rng.choice([0.0, 1e-5, 1e-4])),
        "epochs": SEARCH_EPOCHS,
    }
    started = time.perf_counter()
    _, _, metrics = fit_and_evaluate(config, architecture, train_frame, validation_frame)
    nas_rows.append({
        "trial": trial_number,
        **config,
        "hidden_layers": str(architecture["hidden_layers"]),
        "activation": architecture["activation"],
        "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "parameters": metrics["parameters"],
        "seconds": time.perf_counter() - started,
    })

nas_table = pd.DataFrame(nas_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_nas_row = nas_table.iloc[0].to_dict()
best_nas = {
    "architecture": {
        "hidden_layers": ast.literal_eval(best_nas_row["hidden_layers"]),
        "activation": best_nas_row["activation"],
    },
    "config": {key: best_nas_row[key] for key in ("learning_rate", "batch_size", "weight_decay", "epochs")},
}


### 4.7 Best configuration, performance, and search time

The next cell reports the best NAS architecture, training configuration, validation macro-F1, and total search time after execution.


In [ ]:
# Print the winning NAS architecture and its validation performance.
print("Best NAS architecture:", best_nas["architecture"])
print("Best NAS training configuration:", best_nas["config"])
print("Best validation macro-F1:", best_nas_row["macro_f1"])
print("Total NAS-search time:", nas_table["seconds"].sum())


### 4.8 Search results table

The table below lists every sampled NAS trial.


In [ ]:
# Show every NAS architecture and training configuration that was evaluated.
display(nas_table.round(6))


## Task 5 — Performance comparison and analysis


### 5.1 Compare primary and secondary metrics

The comparison retrains each selected configuration on the same training split and evaluates macro-F1 and balanced accuracy on the same held-out test split.


In [ ]:
# Retrain each selected validation configuration on the same training split, then test once.
def evaluate_selected(name, config, architecture, trials, search_seconds):
    started = time.perf_counter()
    model, _, metrics = fit_and_evaluate(config, architecture, train_frame, test_frame)
    return {
        "method": name,
        "macro_f1": metrics["macro_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "parameters": metrics["parameters"],
        "trials": trials,
        "search_seconds": search_seconds,
        "retrain_seconds": time.perf_counter() - started,
        "config": config,
        "architecture": architecture,
    }


comparison_rows = [
    evaluate_selected("Baseline", BASELINE_CONFIG, BASELINE_ARCHITECTURE, 1, 0.0),
    evaluate_selected(
        "Random search",
        {key: best_random[key] for key in ("learning_rate", "batch_size", "weight_decay", "epochs")},
        BASELINE_ARCHITECTURE,
        RANDOM_TRIALS,
        random_table["seconds"].sum(),
    ),
    evaluate_selected("Bayesian optimisation", best_bayesian, BASELINE_ARCHITECTURE, BAYESIAN_TRIALS, bayesian_table["seconds"].sum()),
    evaluate_selected("Random NAS", best_nas["config"], best_nas["architecture"], NAS_TRIALS, nas_table["seconds"].sum()),
]
comparison = pd.DataFrame(comparison_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)


### 5.2 Search time, trials, and model complexity

The comparison table includes trial count, search time, retraining time, and trainable parameter count.


### 5.3 Summary table and comparison visualisation

The table and figure are generated after the selected configurations are evaluated on the held-out test split.


In [ ]:
# Display the held-out comparison without exposing internal configuration dictionaries.
display(comparison.drop(columns=["config", "architecture"]).round(6))

# Plot both required metrics for the four selected configurations.
ax = comparison.plot.bar(
    x="method", y=["macro_f1", "balanced_accuracy"], ylim=(0, 1), figsize=(9, 4), rot=15
)
ax.set_ylabel("score")
ax.set_title("Held-out performance comparison")
plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=150)
plt.show()


### 5.4 Hyperparameter or architecture effects

Use the generated tables to relate learning rate, regularisation, batch size, and architecture choices to validation score, test score, runtime, and parameter count.


### 5.5 Fairness of the comparison

All methods use the same seed, preprocessing rule, data splits, metric, and epoch budget. Search spaces and trial budgets differ by method, and NAS also changes architecture.


### 5.6 Final model selection and complete configuration

The final method is selected by validation macro-F1, not by held-out test performance. It is then retrained on the combined 80% training and validation pool.


In [ ]:
# Select by validation score, so held-out test scores do not choose the final model.
selection_candidates = [
    {"method": "Baseline", "macro_f1": validation_macro_f1, "config": BASELINE_CONFIG, "architecture": BASELINE_ARCHITECTURE},
    {"method": "Random search", "macro_f1": best_random["macro_f1"], "config": {key: best_random[key] for key in ("learning_rate", "batch_size", "weight_decay", "epochs")}, "architecture": BASELINE_ARCHITECTURE},
    {"method": "Bayesian optimisation", "macro_f1": bayesian_table.iloc[0]["macro_f1"], "config": best_bayesian, "architecture": BASELINE_ARCHITECTURE},
    {"method": "Random NAS", "macro_f1": best_nas_row["macro_f1"], "config": best_nas["config"], "architecture": best_nas["architecture"]},
]
selected = max(selection_candidates, key=lambda candidate: candidate["macro_f1"])
final_config = selected["config"].copy()
final_config["epochs"] = FINAL_EPOCHS
final_architecture = selected["architecture"]
print("Selected method:", selected["method"])
print("Validation macro-F1 used for selection:", selected["macro_f1"])
print("Final configuration:", {"architecture": final_architecture, **final_config})


### 5.7 Class-level evaluation, limitations, and saved model

The final cell reports the class-level test evaluation and saves the model, architecture, feature order, and fitted scaler values to a `.pt` file.


In [ ]:
# Retrain the selected model on the full 80% train-plus-validation pool.
final_model, final_scaler, final_metrics = fit_and_evaluate(
    final_config, final_architecture, train_validation_frame, test_frame
)

# Report class-level test evidence for the saved final model.
report = pd.DataFrame(
    classification_report(
        final_metrics["target"] + 1,
        final_metrics["prediction"] + 1,
        output_dict=True,
        zero_division=0,
    )
).T
print("Final held-out macro-F1:", final_metrics["macro_f1"])
print("Final held-out balanced accuracy:", final_metrics["balanced_accuracy"])
display(report.round(6))

# Save model weights plus every preprocessing value needed for hidden-test prediction.
torch.save(
    {
        "state_dict": final_model.state_dict(),
        "architecture": final_architecture,
        "feature_names": feature_names,
        "continuous_features": continuous_features,
        "scaler_mean": final_scaler.mean_.tolist(),
        "scaler_scale": final_scaler.scale_.tolist(),
        "config": final_config,
    },
    MODEL_PATH,
)
assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0
print(f"Saved final model: {MODEL_PATH}")


## Task 6 — Hidden test


### 6.1 Run the final model on a hidden-test CSV and create Cover_Type predictions

Run `python predict.py data/input.csv data/predictions.csv` from the repository root. The script loads `models/1173808_Assignment2_final.pt`, validates the 14 feature columns, applies the saved scaler, and writes one `Cover_Type` prediction per row.
